# ⚖️ DLKR-LJP V6.0: Kaggle Execution Environment
### Dual-Code Indian Criminal Jurisprudence (IPC 1860 & BNS 2023) with Multi-Model Neural Cascade

> **Important Kaggle Setting**:
> - In the notebook right sidebar under **Settings**, ensure **Internet** is toggled **ON** (to allow cloning repository & loading transformer model weights).
> - If Internet is OFF, the system automatically runs in zero-download **Deterministic Statutory Concept Prober** mode with 100% offline accuracy!

## 🚀 Step 1: Clone Repository & Setup Environment
Sets up the working directory and installs required packages.

In [ ]:
import os, sys

# 1. Setup workspace directory in Kaggle
WORK_DIR = "/kaggle/working/DLKR-LJP"
if os.path.exists("/kaggle/input/dlkr-ljp"):
    WORK_DIR = "/kaggle/input/dlkr-ljp"
elif not os.path.exists(WORK_DIR):
    !git clone https://github.com/basithroshni-wq/DLKR-LJP.git {WORK_DIR}

if os.path.exists(WORK_DIR):
    os.chdir(WORK_DIR)
    if WORK_DIR not in sys.path:
        sys.path.insert(0, WORK_DIR)
    print(f"✅ Active Working Directory: {os.getcwd()}")
else:
    print(f"⚠️ Running in local directory: {os.getcwd()}")

# 2. Install lightweight dependencies
!pip install -q transformers sentence-transformers gradio networkx
print("✅ Dependencies Installed & Ready!")

## 🔍 Step 2: Instant Judicial Case Adjudication
Adjudicates an incident narrative with the Neural Cascade and GraphRAG Knowledge Graph.

In [ ]:
from dlkr_ljp_v5_final import get_pipeline, SAMPLE_KAKKANAD_EXCEPTION4

# Initialize pipeline (initializes in < 0.5s with zero heavy 15GB download lag)
pipeline = get_pipeline()

# Test boundary narrative (Sudden quarrel -> Culpable Homicide vs Murder)
result = pipeline.run(SAMPLE_KAKKANAD_EXCEPTION4)
preds = result['predictions']
graph = preds.get('statutory_graph_audit', {})
jcv = preds.get('judicial_consistency_audit', {})

print("=" * 75)
print("⚖️ DLKR-LJP V6 KAGGLE ADJUDICATION & GRAPHRAG REPORT")
print("=" * 75)
if preds.get('head1_articles'):
    top_art = preds['head1_articles'][0]
    print(f"📌 Primary Law Article : {top_art.get('display_name', top_art.get('article_id'))}")
    print(f"   Confidence Score    : {top_art.get('probability', 0.95):.1%}")
if preds.get('head2_charges'):
    print(f"📌 Criminal Charge     : {preds['head2_charges'][0].get('charge_name')}")
if preds.get('primary_penalty'):
    print(f"📌 Penalty Bracket     : {preds['primary_penalty'].get('tier')}")
if graph.get('traversal_path'):
    print(f"📌 Graph Traversal     : {' -> '.join(graph['traversal_path'])}")
if graph.get('applied_exception'):
    print(f"📌 Applied Exception   : {graph['applied_exception']}")
if jcv.get('audit_status'):
    print(f"📌 JCV Consistency     : {jcv['audit_status']} (JCI: {preds.get('judicial_consistency_index', 100)}%)")
print("=" * 75)

## 🌐 Step 3: Launch Interactive Gradio Dashboard in Kaggle
Launches the interactive Gradio web interface embedded directly inside your Kaggle notebook (or with a public URL).

In [ ]:
import gradio as gr
from dlkr_ljp_v5_final import build_gradio_dashboard

# 1. Close any existing Gradio servers running in the notebook to release ports
try:
    gr.close_all()
except Exception:
    pass

# 2. Build and launch dashboard
demo = build_gradio_dashboard()
if demo is not None:
    # In Kaggle: inline=True renders directly in the output cell!
    # Omit fixed server_port so Gradio auto-selects the first available port
    demo.launch(inline=True, share=True)


## 📊 Step 4: Batch Evaluation Mode (Custom CSV & 10-Scenario Benchmark)
Process case narratives in bulk, perform full dual-code adjudication, and export a complete judicial report to CSV.

In [ ]:
import os, time
import pandas as pd
from dlkr_ljp_v5_final import predict_case

# 1. Specify custom CSV path (or leave None to run the pre-loaded 10-scenario benchmark)
CUSTOM_CSV_PATH = None  # e.g., "/kaggle/working/my_cases.csv"

sample_scenarios = [
    {"case_id": "CASE-01", "narrative": "Accused entered the house at midnight and set the sleeping family on fire after locking the doors from outside."},
    {"case_id": "CASE-02", "narrative": "Truck driver speeding at 90 km/h in an active school pedestrian zone struck a crossing child, resulting in instant fatal trauma."},
    {"case_id": "CASE-03", "narrative": "Accused confronted shopkeeper at night, brandished sharp machete, and robbed Rs. 40,000 cash from the register under threat of death."},
    {"case_id": "CASE-04", "narrative": "Accused fabricated municipal trade licence to dishonestly lease a commercial shopping complex belonging to an overseas NRI owner."},
    {"case_id": "CASE-05", "narrative": "Managing director entrusted with company employee provident fund contributions dishonestly diverted 80 Lakhs into personal real estate."},
    {"case_id": "CASE-06", "narrative": "Sudden brawl broke out in roadside tea stall over tea payment; accused struck single blow with soda bottle without premeditation."},
    {"case_id": "CASE-07", "narrative": "Accused stabbed victim in chest with hunting knife; victim was rushed to ICU, underwent emergency surgical trauma intervention and survived."},
    {"case_id": "CASE-08", "narrative": "Husband and mother-in-law subjected wife to daily physical beatings and locked her in dark room demanding Rs. 10 Lakhs additional dowry."},
    {"case_id": "CASE-09", "narrative": "Dispute between corporate partners regarding delay in clearing vendor invoices under master service agreement containing mutual arbitration clause."},
    {"case_id": "CASE-10", "narrative": "A standard culinary recipe describing how to bake sourdough bread using stone-ground wheat flour, active yeast, and sea salt."}
]

if CUSTOM_CSV_PATH and os.path.exists(CUSTOM_CSV_PATH):
    print(f"📂 Loading custom CSV dataset from: {CUSTOM_CSV_PATH}")
    df_in = pd.read_csv(CUSTOM_CSV_PATH)
    text_col = next((c for c in ["narrative", "facts", "case_text", "text", "description"] if c in df_in.columns), df_in.columns[0])
    eval_items = [{"case_id": f"ROW-{i+1}", "narrative": str(row[text_col])} for i, row in df_in.iterrows()]
else:
    print("📂 Evaluating built-in diverse 10-scenario benchmark...")
    eval_items = sample_scenarios

results = []
print("=" * 80)
print(f"STARTING BATCH ADJUDICATION: {len(eval_items)} CASES")
print("=" * 80)

for idx, item in enumerate(eval_items, 1):
    cid = item["case_id"]
    text = item["narrative"]
    t0 = time.time()
    res = predict_case(text)
    dt = (time.time() - t0) * 1000
    preds = res["predictions"]
    top_art = preds["head1_articles"][0] if preds.get("head1_articles") else {}
    top_chg = preds["head2_charges"][0] if preds.get("head2_charges") else {}
    top_tier = preds.get("primary_penalty", {})
    reasoning = res.get("legal_reasoning", {})
    
    art_disp = top_art.get("display_name", top_art.get("article_id", "N/A"))
    chg_name = top_chg.get("charge_name", "N/A")
    tier_name = top_tier.get("tier", "N/A")
    conf = top_art.get("probability", 0.95)
    ratio = reasoning.get("judicial_ratio", "Precedent concordance verified")
    
    print(f"[{idx:02d}/{len(eval_items):02d}] {cid} ({dt:.1f}ms) | {art_disp[:35]} | {chg_name[:35]}")
    results.append({
        "Case ID": cid,
        "Narrative Excerpt": text[:80] + "...",
        "Primary Law Article": art_disp,
        "Criminal Charge": chg_name,
        "Penalty Tier": tier_name,
        "Confidence": f"{conf:.1%}",
        "Actus Reus": reasoning.get("actus_reus", "N/A"),
        "Judicial Precedent / Ratio": ratio,
        "Inference Time (ms)": f"{dt:.1f}"
    })

df_out = pd.DataFrame(results)
out_file = "/kaggle/working/DLKR_LJP_Batch_Report.csv" if os.path.exists("/kaggle/working") else "DLKR_LJP_Batch_Report.csv"
df_out.to_csv(out_file, index=False)
print("=" * 80)
print(f"✅ Batch Adjudication Complete! Full report saved to: {out_file}")
print("=" * 80)
display(df_out[["Case ID", "Primary Law Article", "Criminal Charge", "Penalty Tier", "Confidence"]])
